### Report purpose

This notebook builds the base analytical table for the project from the twelve monthly NYC yellow taxi files for 2019. It is the foundation of the rest of the pipeline.

The goal here is operational rather than predictive: gather the monthly files, standardize their structure, sample them consistently, and save one reliable master dataset for the later cleaning and modeling stages.

By the end of this notebook we have a single parquet file that is easier to audit, faster to reload, and ready for downstream quality checks.


In [1]:
from pathlib import Path
import pandas as pd

### Paths and run settings

This section prepares the notebook so it can run from different working directories without hardcoded paths. The code searches upward until it finds the project `data` folder, which makes the workflow more portable and easier for a new collaborator to reproduce.

We also define the raw input location, the output location, the sample size per month, and a fixed random seed. The fixed seed matters because it makes the sampled master table reproducible across runs.


In [2]:
# Robust project root (works even if the notebook is executed from notebooks/)
cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_input = project_root / "data" / "raw"
path_output = project_root / "data" / "procesed"   # kept as originally named
path_output.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_MONTH = 1_000_000
RANDOM_STATE = 42

print("project_root:", project_root)
print("path_input:", path_input)
print("path_output:", path_output)

project_root: C:\Users\leodo\Desktop\NYC-Taxi-ML
path_input: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\raw
path_output: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\procesed


### Locating the monthly source files

The raw input data is organized by month, so the first validation step is to confirm that all expected 2019 CSV files are present. The notebook searches for files that match `yellow_tripdata_2019-*.csv`.

If no files are found, the notebook fails early. That is intentional: it is better to stop with a clear setup error than to continue with a partial or misleading dataset.


In [3]:
def find_month_files(input_dir: Path, year: int = 2019) -> list[Path]:
    files = sorted(input_dir.glob(f"yellow_tripdata_{year}-*.csv"))
    if not files:
        raise FileNotFoundError(f"No monthly files found in {input_dir} for year={year}")
    return files

files_2019 = find_month_files(path_input, 2019)
[f.name for f in files_2019][:5], len(files_2019)

(['yellow_tripdata_2019-01.csv',
  'yellow_tripdata_2019-02.csv',
  'yellow_tripdata_2019-03.csv',
  'yellow_tripdata_2019-04.csv',
  'yellow_tripdata_2019-05.csv'],
 12)

### Standardizing the schema

Monthly TLC extracts do not always expose exactly the same set of columns. Before concatenating anything, this notebook computes the union of all columns observed across the 2019 files.

That design choice protects the pipeline in three ways: it preserves all available variables, prevents column-mismatch errors, and ensures the final master table has one consistent schema. Missing columns for a given month are filled with `NA` during alignment.


In [4]:
def get_union_columns(files: list[Path]) -> list[str]:
    union = set()
    for f in files:
        cols = pd.read_csv(f, nrows=0).columns.tolist()
        union.update(cols)
    # stable order: keep TLC-like ordering by sorting
    return sorted(union)

all_cols = get_union_columns(files_2019)
print("Total union columns:", len(all_cols))
all_cols

Total union columns: 18


['DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime',
 'trip_distance']

### Reading, sampling, and aligning each month

Each monthly file goes through the same mini-pipeline: load the CSV, down-sample if it exceeds the target size, align the columns to the common schema, and attach metadata fields such as `year`, `month`, and `source_file`.

The metadata fields are useful for traceability because they preserve the origin of every row. The equal-size monthly sampling also helps maintain seasonal coverage while keeping the dataset manageable for notebook-based analysis.


In [5]:
def month_from_filename(file_path: Path) -> int:
    # yellow_tripdata_2019-01.csv -> 01
    return int(file_path.stem.split("_")[-1].split("-")[1])

def read_and_sample_month(file_path: Path, all_cols: list[str],
                          n: int = 1_000_000, random_state: int = 42) -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)

    # sample (only if needed)
    if len(df) > n:
        df = df.sample(n=n, random_state=random_state)

    # align columns to union (adds missing columns as NA)
    df = df.reindex(columns=all_cols)

    # add metadata
    df["year"] = 2019
    df["month"] = month_from_filename(file_path)
    df["source_file"] = file_path.name

    return df

### Building the master table

After each monthly file has been processed with the same rules, the notebook concatenates them into one master dataframe. This is the first project-wide table with a consistent structure across the full 2019 period.

At this stage the dataset is standardized and balanced by month, but it is not yet fully cleaned for modeling. That work happens in the next notebook.


In [6]:
parts = []
for f in files_2019:
    df_m = read_and_sample_month(f, all_cols, SAMPLE_PER_MONTH, RANDOM_STATE)
    parts.append(df_m)
    print(f"{f.name}: sampled {len(df_m):,} rows | cols={df_m.shape[1]:,}")

master = pd.concat(parts, ignore_index=True)
print("MASTER SHAPE:", master.shape)
master.head()

yellow_tripdata_2019-01.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-02.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-03.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-04.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-05.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-06.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-07.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-08.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-09.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-10.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-11.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-12.csv: sampled 1,000,000 rows | cols=21
MASTER SHAPE: (12000000, 21)


,DOLocationID,PULocationID,RatecodeID,VendorID,congestion_surcharge,extra,fare_amount,improvement_surcharge,mta_tax,passenger_count,...,store_and_fwd_flag,tip_amount,tolls_amount,total_amount,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance,year,month,source_file
0,234,164,1.0,2.0,NaN,0.0,4.5,0.3,0.5,1.0,...,N,1.06,0.0,6.36,2019-01-09 09:31:43,2019-01-09 09:28:06,0.52,2019,1,yellow_tripdata_2019-01.csv
1,230,100,1.0,2.0,NaN,0.0,7.0,0.3,0.5,1.0,...,N,1.95,0.0,9.75,2019-01-02 07:37:09,2019-01-02 07:29:10,1.15,2019,1,yellow_tripdata_2019-01.csv
2,162,140,1.0,2.0,NaN,1.0,10.5,0.3,0.5,1.0,...,N,0.00,0.0,12.30,2019-01-07 16:06:42,2019-01-07 15:55:27,2.44,2019,1,yellow_tripdata_2019-01.csv
3,239,151,1.0,1.0,NaN,0.0,5.5,0.3,0.5,1.0,...,N,1.25,0.0,7.55,2019-01-09 06:56:05,2019-01-09 06:52:41,1.20,2019,1,yellow_tripdata_2019-01.csv
4,260,140,1.0,1.0,NaN,0.0,20.0,0.3,0.5,1.0,...,N,0.00,0.0,20.80,2019-01-17 09:14:37,2019-01-17 08:50:24,4.60,2019,1,yellow_tripdata_2019-01.csv


### Saving the consolidated output

The master table is saved in parquet format because parquet is more storage-efficient and faster to load than CSV for large tabular data.

Persisting the output here is an important workflow choice: later notebooks can start from a stable intermediate artifact instead of repeating the raw-file ingestion process every time.


In [7]:
out_file = path_output / "master_2019_1M_per_month.parquet"
master.to_parquet(out_file, index=False)
print("Saved:", out_file)

Saved: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\procesed\master_2019_1M_per_month.parquet


### Chapter summary

This notebook delivers the project baseline dataset. It does not yet remove implausible trips or engineer predictive features; instead, it creates a reproducible and well-documented starting point for those later steps.

The most important decisions in this stage are: use every 2019 monthly file, sample each month consistently, align all files to a shared schema, preserve row-level traceability, and store the result in parquet for efficient reuse.
